# Revision inicial de datos consolidados

**Catastro de singularidades - CIREN**

Notebook tecnico para iniciar revisiones controladas sobre el Survey123 de origen y el servicio consolidado. Su proposito es observar estructura, volumen y campos clave sin editar entidades, adjuntos ni configuracion del Portal.

> Todas las operaciones incluidas son de solo lectura.

## 1. Preparacion

Antes de ejecutar:

- activar el ambiente de Python de ArcGIS Pro;
- mantener `credenciales.json` en esta carpeta o ajustar `CREDENTIALS_PATH`;
- verificar que el perfil `PORTAL` contenga `url`, `username` y `password`;
- confirmar los Item ID del ambiente que sera revisado.

El archivo real de credenciales esta excluido de Git. Solo se distribuye `credenciales.example.json`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from arcgis.gis import GIS
from IPython.display import display

CREDENTIALS_PATH = Path("./credenciales.json")
CREDENTIALS_PROFILE = "PORTAL"

ITEMS = {
    "survey_feature_service": "3f283dcb6d0f42dead81fc9059509550",
    "target_feature_service": "1a2d5e2632524b709b0007c4c53841c4",
}

## 2. Conexion al Portal

La conexion utiliza credenciales de usuario y contrasena. La contrasena no se imprime ni se incorpora en las salidas del notebook.

In [ ]:
if not CREDENTIALS_PATH.exists():
    raise FileNotFoundError(
        f"No se encontr? {CREDENTIALS_PATH.resolve()}. "
        "Cree el archivo a partir de credenciales.example.json."
    )

with CREDENTIALS_PATH.open(encoding="utf-8") as credentials_file:
    credentials = json.load(credentials_file)[CREDENTIALS_PROFILE]

gis = GIS(
    credentials["url"],
    credentials["username"],
    credentials["password"],
)
print(f"Conexi?n correcta: {gis.url}")

## 3. Revision de la tabla padre del Survey

Se consulta unicamente una muestra de campos de control. La tabla padre concentra los atributos de la inspeccion y la validacion que habilita la consolidacion.

In [ ]:
survey_item = gis.content.get(ITEMS["survey_feature_service"])
if survey_item is None or not survey_item.tables:
    raise ValueError("El Item del Survey no existe o no contiene una tabla padre.")

parent_table = survey_item.tables[0]
parent_result = parent_table.query(
    where="1=1",
    out_fields="globalid,uniquerowid,identificador,validacion",
    return_geometry=False,
)
parent_df = parent_result.sdf

print(f"Survey: {survey_item.title}")
print(f"Tabla padre: {parent_table.properties.name}")
print(f"Registros consultados: {len(parent_df):,}")
display(parent_df.head(10))

if "validacion" in parent_df.columns:
    display(
        parent_df["validacion"]
        .fillna("Sin valor")
        .value_counts(dropna=False)
        .rename_axis("validacion")
        .to_frame("cantidad")
    )

## 4. Revision del servicio consolidado

Primero se presenta el catalogo de capas. Esto permite identificar cual corresponde a puntos y cual a lineas sin depender de una posicion asumida.

In [ ]:
target_item = gis.content.get(ITEMS["target_feature_service"])
if target_item is None or not target_item.layers:
    raise ValueError("El Item destino no existe o no contiene capas.")

layer_catalog = []
for index, layer in enumerate(target_item.layers):
    layer_catalog.append(
        {
            "indice": index,
            "nombre": layer.properties.name,
            "tipo_geometria": layer.properties.geometryType,
            "has_z": bool(layer.properties.get("hasZ", False)),
            "adjuntos": bool(layer.properties.get("hasAttachments", False)),
        }
    )

print(f"Servicio consolidado: {target_item.title}")
display(pd.DataFrame(layer_catalog))

## 5. Muestra de una capa destino

Seleccione el indice informado en el catalogo anterior. La consulta esta limitada a una muestra para mantener una revision rapida y evitar cargar el servicio completo.

In [ ]:
TARGET_LAYER_INDEX = 0
SAMPLE_SIZE = 10

if TARGET_LAYER_INDEX >= len(target_item.layers):
    raise IndexError("TARGET_LAYER_INDEX no corresponde a una capa disponible.")

target_layer = target_item.layers[TARGET_LAYER_INDEX]
target_result = target_layer.query(
    where="1=1",
    out_fields="*",
    return_geometry=True,
    result_record_count=SAMPLE_SIZE,
)
target_sdf = target_result.sdf

print(f"Capa revisada: {target_layer.properties.name}")
print(f"Registros de muestra: {len(target_sdf):,}")
display(target_sdf.head(SAMPLE_SIZE))

## 6. Comprobacion de campos clave

La ultima revision confirma la disponibilidad de los campos utilizados por la consolidacion y permite observar valores de cota sin asumir que existe una fila con indice `0`.

In [ ]:
KEY_FIELDS = ["id_unique", "cota", "cota_manual"]
available_fields = [field for field in KEY_FIELDS if field in target_sdf.columns]
missing_fields = [field for field in KEY_FIELDS if field not in target_sdf.columns]

print(f"Campos disponibles: {available_fields}")
print(f"Campos no disponibles en esta capa: {missing_fields}")

if available_fields and not target_sdf.empty:
    display(target_sdf[available_fields].head(SAMPLE_SIZE))
else:
    print("La muestra no contiene filas o campos clave para presentar.")

## Proximas revisiones sugeridas

A partir de esta base se pueden agregar, en celdas independientes:

1. comparacion de padres validados frente a `id_unique` consolidado;
2. deteccion de duplicados por `id_unique`;
3. identificacion de geometrias huerfanas;
4. control de valores Z en puntos y lineas;
5. revision de consistencia entre cotas, coordenadas y tipo geometrico.

Mantenga estas comprobaciones como consultas de solo lectura. Cualquier correccion de datos debe implementarse y probarse en un flujo separado.